Realizamos las importaciones necesarias:

In [2]:
import os
import csv
import ast
import re
import time
import nltk
import collections
import numpy as np
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support, f1_score
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.base import clone
from joblib import parallel_backend
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification

c:\Users\alexr\miniconda3\envs\lc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fijamos las variables globales del cuaderno:

In [3]:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "etiquetas_test_sin_alias.csv")
EMB_FILE = "embeddings_specter2.npz"
SPECTER2 = "allenai/specter2_base"
SPECTER2_DIR = "specter2"
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

Definimos funciones de carga de datos, tokenización, y cálculo de métricas y estadísticas:

In [4]:
# Función para cargar el corpus:
def load_corpus(filename, delimiter=',', duplicates=False):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Creamos un set para los títulos ya encontrados a fin de evitar duplicados:
        titles = set()

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            title = row[0].strip().lower()
            summary = row[1].strip().lower()

            if not duplicates:
                if title not in titles:
                    titles.add(title)
                else:
                    continue

            papers.append((title, summary))
            terms.append(ast.literal_eval(row[-1]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    probs = {}
    for label, freq in sorted(ct.items(), key=lambda item: item[1], reverse=True):
        prob = freq / len(texts)
        probs[label] = prob
        print(f"{label}: {freq} ({prob:.1%})", end="  ")
    print("\n")
    return probs


try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

## Specter2

### Entrenamiento de una SVM con *embeddings*

Cargamos el modelo (Specter2) y su tokenizador, y fijamos el dispositivo de ejecución:

In [5]:
# Cargamos el modelo y el tokenizador de Specter2:
tokenizer_s2 = AutoTokenizer.from_pretrained(SPECTER2)
model_s2 = AutoModel.from_pretrained(SPECTER2)

# Fijamos el dispositivo de ejecución y establecemos el modelo en modo evaluación:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_cuda = device.type == "cuda"
print("Dispositivo utilizado:", device)
model_s2.to(device)
model_s2.eval();

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5987.74it/s]


Dispositivo utilizado: cuda


Definimos una función para extraer los embeddings de nuestros papers con el modelo congelado:

In [6]:
def get_embeddings(papers, model, tokenizer, batch_size=32):
    embeddings = []

    # Como estamos en inferencia, desactivamos el cálculo de gradientes:
    with torch.no_grad():
        for i in range(0, len(papers), batch_size):

            # Confeccionamos un batch de 32 papers:
            batch_papers = papers[i : i + batch_size]

            # Tokenizamos cada paper:
            inputs = tokenizer(
                batch_papers,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)

            # Recibimos los embeddings asociados a los papers tokenizados (reducimos los parámetros a 16 bits):
            with torch.autocast(
                device_type="cuda" if is_cuda else "cpu",
                dtype=torch.float16 if is_cuda else torch.bfloat16,
                enabled=is_cuda
            ):
                outputs = model(**inputs)

            # Tomamos el embedding del token [CLS] (índice 0) para todo el batch:
            cls_vectors = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_vectors)

    return np.vstack(embeddings)

Cargamos el dataset de entrenamiento y reservamos una partición de validación:

Carga de datos SIN DUPLICADOS:

In [7]:
corpus = {"nd": {}, "d": {}}

In [8]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, ",", False)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{tokenize(title)} {tokenizer_s2.sep_token} {tokenize(summary)}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
train_tok, dev_tok, y_train, y_dev = train_test_split(all_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

# Guardamos los datos sin duplicados en el diccionario 'corpus':
corpus["nd"] = {
    "raw_all": raw_all,
    "y_all": y_all,
    "y_bin": y_bin,
    "all_tok": all_tok,
    "train_tok": train_tok,
    "dev_tok": dev_tok,
    "y_train": y_train,
    "y_dev": y_dev
}

Papers de entrenamiento: 32569
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


Carga de datos CON DUPLICADOS:

In [9]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, duplicates=True)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
title_counts = collections.Counter([title for title, _ in raw_all])
unique_indices = [i for i, (title, _) in enumerate(raw_all) if title_counts[title] == 1]
dup_indices = [i for i, (title, _) in enumerate(raw_all) if title_counts[title] > 1]

n_dev = int(len(all_tok) * DEV_PART)
test_size_unique = n_dev / len(unique_indices)

train_u_idx, dev_idx = train_test_split(
    unique_indices,
    test_size=test_size_unique,
    shuffle=True,
    random_state=SEED
)

# Unimos los duplicados y los únicos restantes para el conjunto de entrenamiento:
train_idx = dup_indices + train_u_idx

# Extraemos las particiones finales:
train_tok = [all_tok[i] for i in train_idx]
y_train = y_bin[train_idx]

dev_tok = [all_tok[i] for i in dev_idx]
y_dev = y_bin[dev_idx]

# Guardamos los datos sin duplicados en el diccionario 'corpus':
corpus["d"] = {
    "raw_all": raw_all,
    "y_all": y_all,
    "y_bin": y_bin,
    "all_tok": all_tok,
    "train_tok": train_tok,
    "dev_tok": dev_tok,
    "y_train": y_train,
    "y_dev": y_dev
}

Papers de entrenamiento: 43555
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


Carga de datos DE TEST:

In [29]:
raw_test, y_test_nobin = load_corpus(TEST_FILE, ";")
print("Papers de test:", len(raw_test))

y_test = mlb.transform(y_test_nobin)

test_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_test]

Papers de test: 4885


### SVM con embeddings congelados de Specter2

Obtenemos los embeddings de los papers de entrenamiento y de test:

In [11]:
# Si ya teníamos un archivo de embeddings guardado, lo cargamos:
if os.path.exists(EMB_FILE):
    data = np.load(EMB_FILE)
    X_train_nd_emb, X_dev_nd_emb = data["train_nd"], data["dev_nd"]

# Si no, los calculamos de cero:
else:
    X_train_nd_emb = get_embeddings(corpus["nd"]["train_tok"], model_s2, tokenizer_s2, batch_size=32)
    X_dev_nd_emb = get_embeddings(corpus["nd"]["dev_tok"], model_s2, tokenizer_s2, batch_size=32)

    # Guardamos los embeddings en un archivo comprimido para no tener que recalcularlos al reiniciar el kernel:
    np.savez_compressed(EMB_FILE, train_nd=X_train_nd_emb, dev_nd=X_dev_nd_emb)

# Normalizamos los embeddings para favorecer la convergencia de SVM:
X_train_nd_emb_norm = normalize(X_train_nd_emb, norm="l2")
X_dev_nd_emb_norm = normalize(X_dev_nd_emb, norm="l2")

Definimos funciones para ajustar los umbrales de decisión y realizar predicciones con dichos umbrales ajustados:

In [12]:
# Definimos una función para ajustar los umbrales de decisión de un clasificador:
def adjust_thresholds(model, X, y, min_th=0.01, max_th=1, n_th=200, alg="rl"):
    if model is not None:
        # Obtenemos las probabilidades para cada clase que ha asociado el modelo a cada paper:
        scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)
    else:
        scores = X

    # Creamos un vector con tantos elementos como clases, inicializado a ceros:
    n_classes = len(mlb.classes_)
    optimal_thresholds = np.zeros(n_classes)

    # Definimos una lista de 'n_th' umbrales de decisión equiespaciados entre 'min_th' y 'max_th':
    thresholds = np.linspace(min_th, max_th, n_th)

    # Iteramos sobre cada clase:
    for c in range(n_classes):
        max_f1_c = -1
        best_th = 0

        # Obtenemos la columna de etiquetas y probabilidades para cada paper asociadas a la clase:
        y_c = y[:, c]
        scores_c = scores[:, c]

        # Iteramos sobre cada umbral:
        for th in thresholds:
            # Obtenemos las predicciones de etiquetas asociadas a ese umbral y calculamos su F1 individual:
            preds_c = (scores_c >= th).astype(int)
            f1_c = f1_score(y_c, preds_c, average="binary", zero_division=0)

            # Si el F1 mejora respecto al que tenía la clase ya, lo guardamos:
            if f1_c > max_f1_c:
                max_f1_c = f1_c
                best_th = th

        # En cada entrada 'i' de 'optimal_thresholds', almacenamos el mejor umbral encontrado para la clase 'c_i':
        optimal_thresholds[c] = best_th

    return optimal_thresholds

In [13]:
# Definimos asimismo una función para obtener las predicciones optimizadas dados los umbrales ajustados:
def predict_with_th(model, X, y, thresholds, verbose=False, alg="rl"):
    if model is not None:
        # Obtenemos las probabilidades originales:
        scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)
    else:
        scores = X

    # Actualizamos las predicciones con los umbrales de decisión que hemos calculado:
    preds = (scores >= thresholds).astype(int)

    # Buscamos los papers que se hayan quedado sin clase:
    zero_rows = np.where(preds.sum(axis=1) == 0)[0] # NumPy devuelve una tupla de 1 elemento, por eso lo extraemos

    # Iteramos sobre cada paper sin clase, obtenemos la clase de mayor probabilidad y se la asignamos:
    for r in zero_rows:
        top_class = np.argmax(scores[r])
        preds[r, top_class] = 1

    if verbose:
        # Consultamos las métricas derivadas de usarlos:
        print(compute_metrics(y, preds))

    return preds

Definimos un SVM a entrenar con los embeddings normalizados y ajustamos su valor C:

In [356]:
svm_s2 = {}
best_c_s2, best_f1_s2 = None, -1
for c in [0.5, 1, 3, 5, 7, 10]:
    # Definimos nuestro SVM a entrenar:
    svm_s2[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual="auto",
            class_weight="balanced",
            max_iter=3000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=8,
    ).fit(X_train_nd_emb_norm, corpus["nd"]["y_train"])

    print(f"== C={c}:")
    metrics = compute_metrics(corpus["nd"]["y_dev"], svm_s2[c].predict(X_dev_nd_emb_norm))
    print(metrics)
    print()

    # Si la F1 macro del modelo actual es la mejor hasta el momento, nos guardamos su C asociada:
    if metrics["f1"] > best_f1_s2:
        best_f1_s2 = metrics["f1"]
        best_c_s2 = c

== C=0.5:
{'accuracy': 0.09905853458862054, 'f1': 0.23959553979365097, 'precision': 0.1786933445174534, 'recall': 0.7421021652602381, 'hamming_loss': 0.07444009122273551}

== C=1:
{'accuracy': 0.11318051575931232, 'f1': 0.24646211686366024, 'precision': 0.1838147485004334, 'recall': 0.6973798358746359, 'hamming_loss': 0.06971083562364774}

== C=3:
{'accuracy': 0.13999181334424887, 'f1': 0.26488437474560583, 'precision': 0.1978794744784195, 'recall': 0.6572143305733598, 'hamming_loss': 0.06302993976960412}

== C=5:
{'accuracy': 0.15431846090871879, 'f1': 0.27721787809965626, 'precision': 0.2072839673383087, 'recall': 0.6336337036478846, 'hamming_loss': 0.06021577685515467}

== C=7:
{'accuracy': 0.16250511665984446, 'f1': 0.2833782411313539, 'precision': 0.2127734342049338, 'recall': 0.6197503382223767, 'hamming_loss': 0.0585199695924215}

== C=10:
{'accuracy': 0.17294310274252966, 'f1': 0.2909860819948023, 'precision': 0.21925096827282423, 'recall': 0.6124291830073693, 'hamming_loss': 0

In [357]:
for c in [6, 8, 9, 12]:
    # Definimos nuestro SVM a entrenar:
    svm_s2[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual="auto",
            class_weight="balanced",
            max_iter=3000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=8,
    ).fit(X_train_nd_emb_norm, corpus["nd"]["y_train"])

    print(f"== C={c}:")
    metrics = compute_metrics(corpus["nd"]["y_dev"], svm_s2[c].predict(X_dev_nd_emb_norm))
    print(metrics)
    print()

    # Si la F1 macro del modelo actual es la mejor hasta el momento, nos guardamos su C asociada:
    if metrics["f1"] > best_f1_s2:
        best_f1_s2 = metrics["f1"]
        best_c_s2 = c

== C=6:
{'accuracy': 0.1588211215718379, 'f1': 0.2797553477489381, 'precision': 0.2096239042833659, 'recall': 0.6217595535142244, 'hamming_loss': 0.05933132565347056}

== C=8:
{'accuracy': 0.167621776504298, 'f1': 0.28707289038955247, 'precision': 0.21595829867597474, 'recall': 0.6171449127537663, 'hamming_loss': 0.05782556575638852}

== C=9:
{'accuracy': 0.17130577159230453, 'f1': 0.2891531855957036, 'precision': 0.21774091141885574, 'recall': 0.6134068360228044, 'hamming_loss': 0.057313899771943164}

== C=12:
{'accuracy': 0.1774457634056488, 'f1': 0.2958371155482394, 'precision': 0.22369970083511567, 'recall': 0.6040180565766731, 'hamming_loss': 0.056195544120226884}



Para el mejor valor de C que hayamos encontrado, ajustamos los umbrales de decisión del modelo y realizamos una predicción:

In [ ]:
s2_th = adjust_thresholds(svm_s2[best_c_s2], X_dev_nd_emb_norm, corpus["nd"]["y_dev"], -2, 2, 500, alg="svm")
preds_test = predict_with_th(svm_s2[best_c_s2], X_dev_nd_emb_norm, corpus["nd"]["y_dev"], s2_th, verbose=True, alg="svm")

{'accuracy': 0.40483012689316417, 'f1': 0.3889134185089011, 'precision': 0.357321369768924, 'recall': 0.47507460336254154, 'hamming_loss': 0.03248348049821648}


### Inferencia *zero-shot* mediante similitud coseno

Empezaremos confeccionando un diccionario con las descripciones de las 28 etiquetas que aparecen en el dataset de entrenamiento, obtenidas directamente de la taxonomía de arXiv:

In [352]:
desc_arxiv = {
    "cs.AI": "Artificial Intelligence: Covers all areas of AI except Vision, Robotics, Machine Learning, Multiagent Systems, and Computation and Language (Natural Language Processing), which have separate subject areas. In particular, includes Expert Systems, Theorem Proving (although this may overlap with Logic in Computer Science), Knowledge Representation, Planning, and Uncertainty in AI. Roughly includes material in General Artificial Intelligence, Applications and Expert Systems, Deduction and Theorem Proving, Knowledge Representation Formalisms and Methods, Problem Solving, Control Methods, and Search, and Distributed Artificial Intelligence.",
    "cs.CL": "Computation and Language: Covers natural language processing. Roughly includes material in Natural Language Processing. Note that work on artificial languages (programming languages, logics, formal systems) that does not explicitly address natural-language issues broadly construed (natural-language processing, computational linguistics, speech, text retrieval, etc.) is not appropriate for this area.",
    "cs.CR": "Cryptography and Security: Covers all areas of cryptography and security including authentication, public key cryptosytems, proof-carrying code, etc. Roughly includes material in Security and Protection, and Data Encryption.",
    "cs.CV": "Computer Vision and Pattern Recognition: Covers image processing, computer vision, pattern recognition, and scene understanding. Roughly includes material in Vision and Scene Understanding, Image Processing and Computer Vision, and Pattern Recognition.",
    "cs.CY": "Computers and Society: Covers impact of computers on society, computer ethics, information technology and public policy, legal aspects of computing, computers and education. Roughly includes material in General Computing Milieux, History of Computing, Computers and Education, Computers and Society, Legal Aspects of Computing, and The Computing Profession.",
    "cs.DC": "Distributed, Parallel, and Cluster Computing: Covers fault-tolerance, distributed algorithms, stabilility, parallel computation, and cluster computing. Roughly includes material in Multiple Data Stream Architectures (Multiprocessors), Parallel Architectures, Distributed Systems, Concurrent Programming, Operating Systems Reliability, Operating Systems Organization and Design, and Data Structures.",
    "cs.GR": "Graphics: Covers all aspects of computer graphics. Roughly includes material in all of Computer Graphics, except that Computational Geometry and Object Modeling is likely to have Computational Geometry as the primary subject area.",
    "cs.HC": "Human-Computer Interaction: Covers human factors, user interfaces, and collaborative computing. Roughly includes material in User/Machine Systems and all of Information Interfaces and Presentation (e.g., HCI), except for Multimedia Information Systems, which is more likely to have Multimedia as the primary subject area.",
    "cs.IR": "Information Retrieval: Covers indexing, dictionaries, retrieval, content and analysis. Roughly includes material in General Information Storage and Retrieval, Content Analysis and Indexing, Information Storage, Information Search and Retrieval, and Systems and Software.",
    "cs.IT": "Information Theory: Covers theoretical and experimental aspects of information theory and coding. Includes material in Coding and Information Theory and intersects with Systems and Information Theory.",
    "cs.LG": "Machine Learning: Papers on all aspects of machine learning research (supervised, unsupervised, reinforcement learning, bandit problems, and so on) including also robustness, explanation, fairness, and methodology. cs.LG is also an appropriate primary category for applications of machine learning methods.",
    "cs.MA": "Multiagent Systems: Covers multiagent systems, distributed artificial intelligence, intelligent agents, coordinated interactions. and practical applications. Roughly covers Distributed Artificial Intelligence.",
    "cs.MM": "Multimedia: Roughly includes material in Multimedia Information Systems.",
    "cs.NE": "Neural and Evolutionary Computing: Covers neural networks, connectionism, genetic algorithms, artificial life, adaptive behavior. Roughly includes some material in Other Architecture Styles, Learning, and Pattern Recognition.",
    "cs.RO": "Robotics: Roughly includes material in Robotics.",
    "cs.SI": "Social and Information Networks: Covers the design, analysis, and modeling of social and information networks, including their applications for on-line information access, communication, and interaction, and their roles as datasets in the exploration of questions in these and other domains, including connections to the social and biological sciences. Analysis and modeling of such networks includes topics in Analysis of Algorithms and Problem Complexity, Discrete Mathematics, Probability and Statistics, Database Management, and Artificial Intelligence; applications in computing include topics in Information Storage and Retrieval, Information Systems Applications, and Information Interfaces and Presentation; and applications at the interface of computing and other disciplines include topics in Administrative Data Processing, Physical Sciences and Engineering, Life and Medical Sciences, Social and Behavioral Sciences, Arts and Humanities, Computer-Aided Engineering, and Computers in Other Systems. Papers on computer communication systems and network protocols (e.g. TCP/IP) are generally a closer fit to the Networking and Internet Architecture (cs.NI) category.",
    "cs.SY": "Systems and Control. This section includes theoretical and experimental research covering all facets of automatic control systems. The section is focused on methods of control system analysis and design using tools of modeling, simulation and optimization. Specific areas of research include nonlinear, distributed, adaptive, stochastic and robust control in addition to hybrid and discrete event systems. Application areas include automotive and aerospace control systems, network control, biological systems, multiagent and cooperative control, robotics, reinforcement learning, sensor networks, control of cyber-physical and energy-related systems, and control of computing systems.",
    "eess.IV": "Image and Video Processing: Theory, algorithms, and architectures for the formation, capture, processing, communication, analysis, and display of images, video, and multidimensional signals in a wide variety of applications. Topics of interest include: mathematical, statistical, and perceptual image and video modeling and representation; linear and nonlinear filtering, de-blurring, enhancement, restoration, and reconstruction from degraded, low-resolution or tomographic data; lossless and lossy compression and coding; segmentation, alignment, and recognition; image rendering, visualization, and printing; computational imaging, including ultrasound, tomographic and magnetic resonance imaging; and image and video analysis, synthesis, storage, search and retrieval.",
    "eess.SP": "Signal Processing: Theory, algorithms, performance analysis and applications of signal and data analysis, including physical modeling, processing, detection and parameter estimation, learning, mining, retrieval, and information extraction. The term \"signal\" includes speech, audio, sonar, radar, geophysical, physiological, (bio-) medical, image, video, and multimodal natural and man-made signals, including communication signals and data. Topics of interest include: statistical signal processing, spectral estimation and system identification; filter design, adaptive filtering / stochastic learning; (compressive) sampling, sensing, and transform-domain methods including fast algorithms; signal processing for machine learning and machine learning for signal processing applications; in-network and graph signal processing; convex and nonconvex optimization methods for signal processing applications; radar, sonar, and sensor array beamforming and direction finding; communications signal processing; low power, multi-core and system-on-chip signal processing; sensing, communication, analysis and optimization for cyber-physical systems such as power grids and the Internet of Things.",
    "eess.SY": "Systems and Control: This section includes theoretical and experimental research covering all facets of automatic control systems. The section is focused on methods of control system analysis and design using tools of modeling, simulation and optimization. Specific areas of research include nonlinear, distributed, adaptive, stochastic and robust control in addition to hybrid and discrete event systems. Application areas include automotive and aerospace control systems, network control, biological systems, multiagent and cooperative control, robotics, reinforcement learning, sensor networks, control of cyber-physical and energy-related systems, and control of computing systems.",
    "math.IT": "Information Theory. Covers theoretical and experimental aspects of information theory and coding.",
    "math.OC": "Optimization and Control: Operations research, linear programming, control theory, systems theory, optimal control, game theory",
    "math.ST": "Statistics Theory: Applied, computational and theoretical statistics: e.g. statistical inference, regression, time series, multivariate analysis, data analysis, Markov chain Monte Carlo, design of experiments, case studies",
    "physics.chem-ph": "Chemical Physics: Experimental, computational, and theoretical physics of atoms, molecules, and clusters - Classical and quantum description of states, processes, and dynamics; spectroscopy, electronic structure, conformations, reactions, interactions, and phases. Chemical thermodynamics. Disperse systems. High pressure chemistry. Solid state chemistry. Surface and interface chemistry.",
    "stat.AP": "Applications: Biology, Education, Epidemiology, Engineering, Environmental Sciences, Medical, Physical Sciences, Quality Control, Social Sciences",
    "stat.ME": "Methodology: Design, Surveys, Model Selection, Multiple Testing, Multivariate Methods, Signal and Image Processing, Time Series, Smoothing, Spatial Statistics, Survival Analysis, Nonparametric and Semiparametric Methods",
    "stat.ML": "Machine Learning: Covers machine learning papers (supervised, unsupervised, semi-supervised learning, graphical models, reinforcement learning, bandits, high dimensional inference, etc.) with a statistical or theoretical grounding",
    "stat.TH": "Statistics Theory. Asymptotics, Bayesian Inference, Decision Theory, Estimation, Foundations, Inference, Testing."
}

# Confeccionamos una lista de las descripciones siguiendo el orden de 'mlb.classes_':
desc_list = [desc_arxiv[c] for c in mlb.classes_]

Obtenemos los embeddings de las descripciones y los normalizamos:

In [360]:
# Si ya los teníamos guardados, los cargamos:
with np.load(EMB_FILE) as data:
    if "tags" in data:
        desc_emb = data["tags"]

    # Si no, los calculamos:
    else:
        desc_emb = get_embeddings(desc_list, model_s2, tokenizer_s2, batch_size=32)
        dict_data = dict(data)
        dict_data["tags"] = desc_emb
        np.savez_compressed(EMB_FILE, **dict_data)

# Normalizamos los embeddings:
desc_emb_norm = normalize(desc_emb, norm="l2")

Obtenemos la matriz de similitud semática entre los embeddings de dev y los de las descripciones:

In [361]:
# Multiplicación matricial: (N_test, 768) @ (768, 28) -> (N_test, 28)
sim_dev = np.dot(X_dev_nd_emb_norm, desc_emb_norm.T)

In [362]:
sim_dev[0]

array([0.8468047 , 0.82376677, 0.8087134 , 0.86562127, 0.7915735 ,
       0.8032651 , 0.820219  , 0.8433177 , 0.85477245, 0.8540213 ,
       0.8241627 , 0.82169974, 0.88596547, 0.841033  , 0.8188612 ,
       0.8047324 , 0.8001187 , 0.85721457, 0.8138452 , 0.80027956,
       0.8457165 , 0.7845862 , 0.8122151 , 0.7468874 , 0.76593643,
       0.78946066, 0.84885645, 0.79975516], dtype=float32)

Como todos los valores se encuentran en el rango [0.79, 0.90], vamos a **estandarizarlos** (restarles la media y dividirlos entre la desviación típica) para estudiar cuáles destacan especialmente:

In [365]:
# Sumamos un pequeño epsilon a la desviación típica para evitar divisiones por cero:
sim_dev_std = (sim_dev - sim_dev.mean(axis=0)) / (sim_dev.std(axis=0) + 1e-8)

In [366]:
sim_dev_std[0]

array([-0.2735493 ,  0.33617342, -0.31571344, -0.05062459,  0.08371391,
       -0.88061905, -0.96146226,  0.87683815,  0.34799117,  0.4529527 ,
       -0.984817  , -0.57398814,  2.8501003 , -0.54586154, -1.0421333 ,
       -0.7052595 , -1.1852018 ,  0.6400573 , -1.1606853 , -1.2012115 ,
        0.13301663, -1.061796  , -0.9836612 , -1.6792047 , -1.6749371 ,
       -1.602559  , -0.737544  , -1.6753134 ], dtype=float32)

Trataremos esta matriz tipificada igual que la función de decisión de SVM, y ajustaremos los umbrales de decisión para cada clase (que en este caso se corresponderan al mínimo número de desviaciones típicas que debe alejarse el Z-score de la clase respecto de la media) a fin de maximizar el F1 en validación:

In [ ]:
sim_th = adjust_thresholds(None, sim_dev_std, corpus["nd"]["y_dev"], -3, 4, 500)
_ = predict_with_th(None, sim_dev_std, corpus["nd"]["y_dev"], sim_th, verbose=True)

{'accuracy': 0.2652476463364716, 'f1': 0.23445957929824085, 'precision': 0.1902548351604041, 'recall': 0.44420349177204266, 'hamming_loss': 0.053629904683936615}


## Specter2 fine-tuneado

In [14]:
def predict_s2(model_name, X):
    path = os.path.join(SPECTER2_DIR, model_name)
    model = AutoModelForSequenceClassification.from_pretrained(path)
    model.to(device)
    model.eval()

    # Definimos el tamaño del lote para evitar saturar la memoria de la tarjeta gráfica:
    BATCH_SIZE = 32
    all_probs = []

    # Iteramos sobre los textos por lotes para extraer las predicciones sin calcular gradientes:
    with torch.no_grad():
        for i in range(0, len(X), BATCH_SIZE):
            batch_texts = X[i : i + BATCH_SIZE]
            
            # Tokenizamos el lote actual limitando la longitud máxima a 384 tokens:
            encoding = tokenizer_s2(batch_texts, truncation=True, padding="max_length", max_length=384, return_tensors="pt").to(device)
            
            # Pasamos los tensores por el modelo:
            outputs = model(**encoding)
            
            # Aplicamos la función sigmoide a los logits para obtener probabilidades continuas:
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.append(probs)

    # Apilamos todas las probabilidades extraídas en una única matriz:
    return np.vstack(all_probs)

In [21]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, duplicates=True)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
_, dev_tok_leak, _, y_dev_leak = train_test_split(all_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

Papers de entrenamiento: 43555
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


In [23]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, duplicates=False)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
_, dev_tok_nodup, _, y_dev_nodup = train_test_split(all_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

Papers de entrenamiento: 32569
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


In [24]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, duplicates=True)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
title_counts = collections.Counter([title for title, _ in raw_all])
unique_indices = [i for i, (title, _) in enumerate(raw_all) if title_counts[title] == 1]

n_dev = int(len(all_tok) * DEV_PART)
test_size_unique = n_dev / len(unique_indices)

_, dev_idx = train_test_split(
    unique_indices,
    test_size=test_size_unique,
    shuffle=True,
    random_state=SEED
)

dev_tok_dup = [all_tok[i] for i in dev_idx]
y_dev_dup = y_bin[dev_idx]

Papers de entrenamiento: 43555
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


In [337]:
# Mapeo de cada modelo con su respectivo conjunto de desarrollo:
dev_data = {
    "no_duplicados": (dev_tok_nodup, y_dev_nodup),
    "duplicados": (dev_tok_dup, y_dev_dup),
    "duplicados_data_leakage": (dev_tok_leak, y_dev_leak),
}

for name in ["no_duplicados", "duplicados", "duplicados_data_leakage"]:
    current_dev_tok, current_y_dev = dev_data[name]
    
    # Inferencia en su propio dev:
    probs_dev = predict_s2(name, current_dev_tok)
    dev_th = adjust_thresholds(None, probs_dev, current_y_dev)
    preds_dev_bin = (probs_dev >= 0.5).astype(int)

    # Inferencia en test (test_tok e y_test sí son comunes para todos):
    probs_test = predict_s2(name, test_tok)
    preds_test_bin = (probs_test >= 0.5).astype(int)

    print(f"== {name}:")
    print("Dev - Sin ajuste:")
    print(compute_metrics(current_y_dev, preds_dev_bin))
    print("Dev - Con ajuste:")
    _ = predict_with_th(None, probs_dev, current_y_dev, dev_th, True)
    
    print("Test - Sin ajuste:")
    print(compute_metrics(y_test, preds_test_bin))
    print("Test - Con ajuste:")
    _ = predict_with_th(None, probs_test, y_test, dev_th, True)
    print()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 536.04it/s]


== no_duplicados:
Dev - Sin ajuste:
{'accuracy': 0.5153499795333606, 'f1': 0.20844140711477063, 'precision': 0.3197870908722719, 'recall': 0.1850917124934254, 'hamming_loss': 0.024516110168995964}
Dev - Con ajuste:
{'accuracy': 0.45006139991813343, 'f1': 0.3742629065416211, 'precision': 0.3526436080830685, 'recall': 0.4766999655931162, 'hamming_loss': 0.031094672826150516}
Test - Sin ajuste:
{'accuracy': 0.3887410440122825, 'f1': 0.13216009951422406, 'precision': 0.14436710622035656, 'recall': 0.15286457551255644, 'hamming_loss': 0.03166398596286007}
Test - Con ajuste:
{'accuracy': 0.2556806550665302, 'f1': 0.19910132368355327, 'precision': 0.16492716252427303, 'recall': 0.3330592362782988, 'hamming_loss': 0.044794560608276066}



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 694.44it/s]


== duplicados:
Dev - Sin ajuste:
{'accuracy': 0.5276289606612582, 'f1': 0.1740382765343506, 'precision': 0.24898636578543473, 'recall': 0.1566181434420583, 'hamming_loss': 0.024190374144453436}
Dev - Con ajuste:
{'accuracy': 0.4403796112046533, 'f1': 0.35693807699500224, 'precision': 0.3648516647542877, 'recall': 0.40850615598531104, 'hamming_loss': 0.03199689488530756}
Test - Sin ajuste:
{'accuracy': 0.39754350051177073, 'f1': 0.11643883984142504, 'precision': 0.126638936377263, 'recall': 0.13547889677741412, 'hamming_loss': 0.03088170785202515}
Test - Con ajuste:
{'accuracy': 0.23111566018423746, 'f1': 0.1930236463610925, 'precision': 0.16555085444718684, 'recall': 0.302866433314359, 'hamming_loss': 0.04776283082321977}



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 613.13it/s]


== duplicados_data_leakage:
Dev - Sin ajuste:
{'accuracy': 0.6385062748699112, 'f1': 0.4023030229850315, 'precision': 0.5602483715850336, 'recall': 0.3698444305332212, 'hamming_loss': 0.018010188464733918}
Dev - Con ajuste:
{'accuracy': 0.623201714110805, 'f1': 0.5343991777523992, 'precision': 0.6197718917586361, 'recall': 0.5331897013114925, 'hamming_loss': 0.019004984914075825}
Test - Sin ajuste:
{'accuracy': 0.34431934493346983, 'f1': 0.1630259150818784, 'precision': 0.16063878735964193, 'recall': 0.19721306936095478, 'hamming_loss': 0.03504898376955695}
Test - Con ajuste:
{'accuracy': 0.31893551688843397, 'f1': 0.18177286537777768, 'precision': 0.16793721538157796, 'recall': 0.24268869441157945, 'hamming_loss': 0.037936832870302674}



In [30]:
preds_s2_dev = predict_s2("no_duplicados", dev_tok_nodup)
s2_th = adjust_thresholds(None, preds_s2_dev, y_dev_nodup)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 874.12it/s]


In [33]:
preds_s2_test = predict_s2("no_duplicados", test_tok)
preds_test_bin = predict_with_th(None, preds_s2_test, y_test, s2_th, True)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 647.73it/s]


{'accuracy': 0.2556806550665302, 'f1': 0.19910132368355327, 'precision': 0.16492716252427303, 'recall': 0.3330592362782988, 'hamming_loss': 0.044794560608276066}


In [40]:
import pandas as pd

# 1. Cargar el archivo de test usando su separador:
df_test = pd.read_csv("data/dataset_test.csv", sep=",")

# 2. Convertir y reemplazar la última columna:
pred_labels = [str(list(tags)) for tags in mlb.inverse_transform(preds_test_bin)]
df_test.iloc[:, -1] = pred_labels

# 3. Renombrar la última columna a "terms":
cols = df_test.columns.tolist()
cols[-1] = "terms"
df_test.columns = cols

# 4. Guardar el archivo resultante:
df_test.to_csv("test_con_predicciones.csv", sep=",", index=False, encoding="utf-8")

## Specter2 fine-tuneado con SVM

In [15]:
# Cargamos el modelo fine-tuneado como modelo base (ignorando su cabeza de clasificación):
path_s2_ft = os.path.join(SPECTER2_DIR, "no_duplicados")
model_s2_ft = AutoModel.from_pretrained(path_s2_ft)
model_s2_ft.to(device)
model_s2_ft.eval()

# Extraemos los nuevos embeddings de entrenamiento, desarrollo y test:
print("Extrayendo embeddings con modelo fine-tuneado...")
X_train_ft_emb = get_embeddings(corpus["nd"]["train_tok"], model_s2_ft, tokenizer_s2, batch_size=32)
X_dev_ft_emb = get_embeddings(corpus["nd"]["dev_tok"], model_s2_ft, tokenizer_s2, batch_size=32)
X_test_ft_emb = get_embeddings(test_tok, model_s2_ft, tokenizer_s2, batch_size=32)

# Normalizamos los nuevos embeddings:
X_train_ft_norm = normalize(X_train_ft_emb, norm="l2")
X_dev_ft_norm = normalize(X_dev_ft_emb, norm="l2")
X_test_ft_norm = normalize(X_test_ft_emb, norm="l2")

# Buscamos el mejor valor de C para la SVM sobre estas representaciones mejoradas:
print("\nEntrenando SVM...")
svm_s2_ft = {}
best_c_ft, best_f1_ft = None, -1

for c in [0.5, 1, 3, 5, 7, 10]:
    svm_s2_ft[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual="auto",
            class_weight="balanced",
            max_iter=3000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=8,
    ).fit(X_train_ft_norm, corpus["nd"]["y_train"])

    print(f"== C={c}:")
    metrics_dev = compute_metrics(corpus["nd"]["y_dev"], svm_s2_ft[c].predict(X_dev_ft_norm))
    print(metrics_dev)
    
    if metrics_dev["f1"] > best_f1_ft:
        best_f1_ft = metrics_dev["f1"]
        best_c_ft = c

print(f"\nMejor C encontrado: {best_c_ft}")

# Ajustamos los umbrales de decisión de la mejor SVM usando el conjunto de desarrollo:
print("\nAjustando umbrales y evaluando en test...")
svm_ft_best = svm_s2_ft[best_c_ft]
th_ft = adjust_thresholds(svm_ft_best, X_dev_ft_norm, corpus["nd"]["y_dev"], -2, 2, 500, alg="svm")

# Realizamos la predicción final en el conjunto de test con los umbrales óptimos:
preds_test_ft = predict_with_th(svm_ft_best, X_test_ft_norm, y_test, th_ft, verbose=True, alg="svm")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1326.75it/s]
[transformers] BertModel LOAD REPORT from: specter2\no_duplicados
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extrayendo embeddings con modelo fine-tuneado...

Entrenando SVM...
== C=0.5:
{'accuracy': 0.19361440851412198, 'f1': 0.2938418616490578, 'precision': 0.22196579142350648, 'recall': 0.6398628258844815, 'hamming_loss': 0.05298666744634817}
== C=1:
{'accuracy': 0.21612771182971757, 'f1': 0.3038989249569995, 'precision': 0.23163023012460973, 'recall': 0.6062753442885226, 'hamming_loss': 0.04981433834278697}
== C=3:
{'accuracy': 0.2468276708964388, 'f1': 0.3281685404294317, 'precision': 0.2575884981424622, 'recall': 0.5529512020752133, 'hamming_loss': 0.046232676451669494}
== C=5:
{'accuracy': 0.26033565288579613, 'f1': 0.32961171859850286, 'precision': 0.2630473090733664, 'recall': 0.5302682597188189, 'hamming_loss': 0.04492427343430209}
== C=7:
{'accuracy': 0.26811297584936555, 'f1': 0.32813974955565994, 'precision': 0.2657291579443287, 'recall': 0.5127239332289938, 'hamming_loss': 0.0442225600842056}
== C=10:
{'accuracy': 0.2758902988129349, 'f1': 0.3350350920228244, 'precision': 0.2760

In [16]:
preds_test_direct = svm_ft_best.predict(X_test_ft_norm)
print("Test - SVM directa (sin sobreajuste de umbrales):")
print(compute_metrics(y_test, preds_test_direct))

Test - SVM directa (sin sobreajuste de umbrales):
{'accuracy': 0.14370522006141248, 'f1': 0.17688760732342165, 'precision': 0.14361927838939112, 'recall': 0.3426061676095869, 'hamming_loss': 0.05511770726714432}


In [17]:
EMB_FT_FILE = "embeddings_specter2_ft.npz"

# 1. Preparamos los textos igual que en el fine-tuning (sin filtrar con tokenize):
raw_nd, y_nd = load_corpus(TRAIN_FILE, duplicates=False)
y_nd_bin = mlb.fit_transform(y_nd)
all_tok_clean = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_nd]

train_tok_ft, dev_tok_ft, y_train_ft, y_dev_ft = train_test_split(
    all_tok_clean, y_nd_bin, test_size=DEV_PART, shuffle=True, random_state=SEED
)

raw_test, y_test_nobin = load_corpus(TEST_FILE, ";")
y_test = mlb.transform(y_test_nobin)
test_tok_ft = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_test]

# 2. Extraemos o cargamos de disco los embeddings del modelo fine-tuneado:
if os.path.exists(EMB_FT_FILE):
    print("Cargando embeddings fine-tuneados desde disco...")
    data_ft = np.load(EMB_FT_FILE)
    X_train_ft, X_dev_ft, X_test_ft = data_ft["train"], data_ft["dev"], data_ft["test"]
else:
    print("Extrayendo embeddings con el modelo fine-tuneado y guardando en disco...")
    path_s2_ft = os.path.join(SPECTER2_DIR, "no_duplicados")
    model_s2_ft = AutoModel.from_pretrained(path_s2_ft).to(device)
    model_s2_ft.eval()

    X_train_ft = get_embeddings(train_tok_ft, model_s2_ft, tokenizer_s2, batch_size=32)
    X_dev_ft = get_embeddings(dev_tok_ft, model_s2_ft, tokenizer_s2, batch_size=32)
    X_test_ft = get_embeddings(test_tok_ft, model_s2_ft, tokenizer_s2, batch_size=32)
    np.savez_compressed(EMB_FT_FILE, train=X_train_ft, dev=X_dev_ft, test=X_test_ft)

# 3. Probamos en validación sin/con normalización y con valores de C pequeños (evitando sobreajuste):
best_cfg = None
best_f1_dev = -1
best_model = None

for norm_mode in ["raw", "l2"]:
    X_tr = normalize(X_train_ft, norm="l2") if norm_mode == "l2" else X_train_ft
    X_dv = normalize(X_dev_ft, norm="l2") if norm_mode == "l2" else X_dev_ft

    for cw in [None, "balanced"]:
        for c in [0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0]:
            clf = OneVsRestClassifier(
                LinearSVC(
                    C=c,
                    dual="auto",
                    class_weight=cw,
                    max_iter=3000,
                    tol=1e-3,
                    random_state=SEED,
                ),
                n_jobs=8,
            ).fit(X_tr, y_train_ft)

            # Obtenemos los márgenes y aseguramos al menos 1 etiqueta por paper (argmax):
            scores_dv = clf.decision_function(X_dv)
            preds_dv = (scores_dv >= 0.0).astype(int)
            for r in np.where(preds_dv.sum(axis=1) == 0)[0]:
                preds_dv[r, np.argmax(scores_dv[r])] = 1

            f1_dv = f1_score(y_dev_ft, preds_dv, average="macro", zero_division=0)
            if f1_dv > best_f1_dev:
                best_f1_dev = f1_dv
                best_cfg = (norm_mode, cw, c)
                best_model = clf

norm_mode, best_cw, best_c = best_cfg
print(f"Mejor configuración en Dev -> Modo: {norm_mode} | class_weight: {best_cw} | C: {best_c} | F1 Dev base: {best_f1_dev:.4f}")

# 4. Preparamos las matrices de dev y test según el mejor modo encontrado:
X_dv_best = normalize(X_dev_ft, norm="l2") if norm_mode == "l2" else X_dev_ft
X_ts_best = normalize(X_test_ft, norm="l2") if norm_mode == "l2" else X_test_ft

scores_dev = best_model.decision_function(X_dv_best)
scores_test = best_model.decision_function(X_ts_best)

# 5. Ajuste A: Umbral global único (cero sobreajuste entre dev y test):
best_global_th, best_global_f1 = 0.0, -1
for th in np.linspace(-0.8, 0.8, 161):
    p_dv = (scores_dev >= th).astype(int)
    for r in np.where(p_dv.sum(axis=1) == 0)[0]:
        p_dv[r, np.argmax(scores_dev[r])] = 1
    f1_val = f1_score(y_dev_ft, p_dv, average="macro", zero_division=0)
    if f1_val > best_global_f1:
        best_global_f1 = f1_val
        best_global_th = th

# 6. Ajuste B: Umbrales por clase corregidos (sin el bug de max_f1_c = -1 y amortiguados):
n_classes = len(mlb.classes_)
class_ths = np.full(n_classes, best_global_th)
grid_th = np.linspace(-0.6, 1.2, 181)

for c in range(n_classes):
    y_c = y_dev_ft[:, c]
    sc_c = scores_dev[:, c]
    # Inicializamos con el F1 del umbral global para cambiarlo solo si realmente mejora:
    max_f1_c = f1_score(y_c, (sc_c >= best_global_th).astype(int), average="binary", zero_division=0)
    best_th_c = best_global_th

    for th in grid_th:
        f1_c = f1_score(y_c, (sc_c >= th).astype(int), average="binary", zero_division=0)
        if f1_c > max_f1_c:
            max_f1_c = f1_c
            best_th_c = th

    # Amortiguamos (50% umbral de clase + 50% umbral global) para no sobreajustar en clases raras:
    class_ths[c] = 0.5 * best_th_c + 0.5 * best_global_th

# 7. Evaluamos las opciones en Test:
print("\n=== RESULTADOS EN TEST ===")
print("1) Test - Corte directo en 0.0 (con rescate argmax):")
_ = predict_with_th(None, scores_test, y_test, 0.0, verbose=True)

print(f"\n2) Test - Con umbral global óptimo ({best_global_th:.3f}):")
_ = predict_with_th(None, scores_test, y_test, best_global_th, verbose=True)

print("\n3) Test - Con umbrales por clase amortiguados:")
preds_test_svm_ft = predict_with_th(None, scores_test, y_test, class_ths, verbose=True)

Extrayendo embeddings con el modelo fine-tuneado y guardando en disco...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 593.67it/s]
[transformers] BertModel LOAD REPORT from: specter2\no_duplicados
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 